# CheXmask 다운로드 + U-Net 학습 데이터 준비

**목적**: PhysioNet CheXmask 데이터셋에서 폐/심장 세그멘테이션 마스크를 다운로드하고, U-Net 학습용으로 전처리

**데이터 흐름**:
```
PhysioNet (Preprocessed/MIMIC-CXR-JPG.csv, 4.4GB)
  → wget 다운로드 (SageMaker 노트북)
  → RLE 디코딩 + p10 필터 + 512x512 리사이즈
  → NPZ 파일로 저장 (train/validate/test)
  → S3 업로드
```

**CheXmask CSV 포맷**:
| 컬럼 | 설명 |
|------|------|
| ImageID | MIMIC-CXR 이미지 참조 |
| Dice RCA (Max/Mean) | 마스크 품질 지표 |
| Left Lung / Right Lung / Heart | RLE 인코딩 마스크 |
| Height / Width | 디코딩에 필요한 해상도 |

**예상 소요**: 다운로드 ~10분 + 전처리 ~20분 + S3 업로드 ~5분

## Step 1: 환경 설정

In [ ]:
import os
import subprocess
import time
import numpy as np
import pandas as pd

# 작업 디렉토리
WORK_DIR = '/home/ec2-user/SageMaker/unet_data'
os.makedirs(WORK_DIR, exist_ok=True)

# S3 설정
BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
S3_DATA_PREFIX = 'data/unet_masks'

# 파일 경로
CHEXMASK_CSV = os.path.join(WORK_DIR, 'MIMIC-CXR-JPG.csv')
SPLIT_CSV = os.path.join(WORK_DIR, 'mimic-cxr-2.0.0-split.csv')
OUTPUT_DIR = os.path.join(WORK_DIR, 'masks_512')

print(f"작업 디렉토리: {WORK_DIR}")
print(f"S3 버킷: s3://{BUCKET}/{S3_DATA_PREFIX}/")
print(f"출력 디렉토리: {OUTPUT_DIR}")

## Step 2: CheXmask CSV 다운로드 (~4.4 GB)

PhysioNet에서 **Preprocessed** 버전 다운로드 (1024x1024 균일 해상도).
OriginalResolution(13GB)보다 3배 작고, 이미 균일 해상도라 전처리 부담 적음.

> **주의**: 4.4GB 파일이므로 SageMaker 노트북 디스크 여유 확인 (최소 10GB 필요)

In [ ]:
# 디스크 여유 확인
!df -h /home/ec2-user/SageMaker/

In [ ]:
%%time

CHEXMASK_URL = (
    "https://physionet.org/files/chexmask-cxr-segmentation-data/1.0.0/"
    "Preprocessed/MIMIC-CXR-JPG.csv"
)

if os.path.exists(CHEXMASK_CSV):
    size_gb = os.path.getsize(CHEXMASK_CSV) / (1024**3)
    print(f"이미 다운로드됨: {size_gb:.2f} GB")
else:
    print(f"다운로드 시작: {CHEXMASK_URL}")
    print("약 4.4 GB — 10분 정도 소요됩니다...")
    !wget -O {CHEXMASK_CSV} --progress=bar:force "{CHEXMASK_URL}"
    
    size_gb = os.path.getsize(CHEXMASK_CSV) / (1024**3)
    print(f"\n다운로드 완료: {size_gb:.2f} GB")

## Step 3: Split CSV 준비

MIMIC-CXR split 정보 (train/validate/test 구분). 이전에 S3에 올려둔 CSV 사용.

In [ ]:
if not os.path.exists(SPLIT_CSV):
    print("Split CSV 다운로드 (S3)...")
    !aws s3 cp s3://{BUCKET}/data/mimic-cxr-csv/mimic-cxr-2.0.0-split.csv {SPLIT_CSV}
else:
    print(f"이미 존재: {SPLIT_CSV}")

# 확인
split_df = pd.read_csv(SPLIT_CSV)
print(f"\nSplit CSV: {len(split_df):,}행")
print(split_df['split'].value_counts())

## Step 4: CheXmask CSV 구조 확인

다운로드한 CSV의 컬럼과 샘플 데이터를 확인. RLE 포맷이 어떻게 생겼는지 미리 파악.

In [ ]:
%%time

# 4.4GB CSV → 첫 5행만 로드하여 구조 확인
df_sample = pd.read_csv(CHEXMASK_CSV, nrows=5)

print("=== 컬럼 목록 ===")
for i, col in enumerate(df_sample.columns):
    print(f"  [{i}] {col} (dtype: {df_sample[col].dtype})")

print(f"\n=== 첫 번째 행 ===")
row = df_sample.iloc[0]
print(f"ImageID: {row.iloc[0]}")
print(f"Height: {row['Height']}, Width: {row['Width']}")

# RLE 샘플 (앞 100자만)
for col in ['Left Lung', 'Right Lung', 'Heart']:
    if col in df_sample.columns:
        rle_val = str(row[col])
        print(f"\n{col} RLE (앞 100자): {rle_val[:100]}...")
        # RLE 쌍 개수
        pairs = len(rle_val.split()) // 2
        print(f"  RLE 쌍 수: {pairs}")

# Dice RCA 컬럼 확인
for col in df_sample.columns:
    if 'dice' in col.lower():
        print(f"\n{col}: {row[col]}")

## Step 5: RLE 디코딩 함수 정의

CheXmask 공식 코드 기반 RLE 디코딩.
- RLE 포맷: `"start1 len1 start2 len2 ..."` (1-indexed, 공백 구분)
- `start -= 1`로 0-indexed 변환 후 mask 배열에 채움

In [ ]:
from PIL import Image

def rle_to_mask(rle_string, height, width):
    """
    RLE 문자열 → 바이너리 마스크 (H x W, uint8)
    
    Args:
        rle_string: "start1 len1 start2 len2 ..." (1-indexed, 공백 구분)
        height: 마스크 높이
        width: 마스크 너비
    Returns:
        np.ndarray: shape (height, width), dtype uint8, 값 0 또는 1
    """
    if pd.isna(rle_string) or str(rle_string).strip() in ('', 'nan'):
        return np.zeros((height, width), dtype=np.uint8)
    
    runs = np.array([int(x) for x in str(rle_string).split()])
    starts = runs[0::2]    # 짝수 인덱스: 시작 위치 (1-indexed)
    lengths = runs[1::2]   # 홀수 인덱스: 연속 길이
    
    mask = np.zeros(height * width, dtype=np.uint8)
    for start, length in zip(starts, lengths):
        start -= 1  # 1-indexed → 0-indexed
        mask[start:start + length] = 1
    
    return mask.reshape((height, width))


def decode_masks(row, target_size=512):
    """
    한 행에서 좌폐/우폐/심장 3채널 마스크 추출 + 리사이즈
    
    Returns:
        combined: 0=배경, 1=좌폐, 2=우폐, 3=심장
        개별 마스크: left_lung, right_lung, heart
    """
    h, w = int(row['Height']), int(row['Width'])
    
    left_lung  = rle_to_mask(row.get('Left Lung', ''), h, w)
    right_lung = rle_to_mask(row.get('Right Lung', ''), h, w)
    heart      = rle_to_mask(row.get('Heart', ''), h, w)
    
    # 리사이즈 (Nearest Neighbor — 마스크는 보간하면 안 됨)
    if h != target_size or w != target_size:
        left_lung  = np.array(Image.fromarray(left_lung).resize((target_size, target_size), Image.NEAREST))
        right_lung = np.array(Image.fromarray(right_lung).resize((target_size, target_size), Image.NEAREST))
        heart      = np.array(Image.fromarray(heart).resize((target_size, target_size), Image.NEAREST))
    
    # Combined: 겹치는 영역은 심장 우선
    combined = np.zeros((target_size, target_size), dtype=np.uint8)
    combined[left_lung > 0] = 1
    combined[right_lung > 0] = 2
    combined[heart > 0] = 3
    
    return combined, left_lung, right_lung, heart

print("RLE 디코딩 함수 정의 완료")

## Step 6: RLE 디코딩 테스트 (샘플 1장)

첫 번째 행으로 디코딩이 정상 작동하는지 확인. 마스크가 제대로 그려지는지 시각화.

In [ ]:
import matplotlib.pyplot as plt

# 샘플 행 디코딩
row = df_sample.iloc[0]
combined, left_lung, right_lung, heart = decode_masks(row, target_size=512)

print(f"ImageID: {row.iloc[0]}")
print(f"원본 해상도: {int(row['Height'])} x {int(row['Width'])}")
print(f"리사이즈: 512 x 512")
print(f"좌폐 픽셀: {left_lung.sum():,}")
print(f"우폐 픽셀: {right_lung.sum():,}")
print(f"심장 픽셀: {heart.sum():,}")
print(f"Combined 클래스 분포: {np.bincount(combined.flatten(), minlength=4)}")

# 시각화
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(combined, cmap='nipy_spectral', vmin=0, vmax=3)
axes[0].set_title('Combined\n(0:BG, 1:LL, 2:RL, 3:Heart)')

axes[1].imshow(left_lung, cmap='gray')
axes[1].set_title('Left Lung')

axes[2].imshow(right_lung, cmap='gray')
axes[2].set_title('Right Lung')

axes[3].imshow(heart, cmap='gray')
axes[3].set_title('Heart')

for ax in axes:
    ax.axis('off')

plt.suptitle(f"RLE Decode Test — {row.iloc[0]}", fontsize=12)
plt.tight_layout()
plt.show()

## Step 7: 전체 CSV 로드 + p10 필터링

4.4GB CSV 전체를 로드한 뒤:
1. **품질 필터**: Dice RCA Mean ≥ 0.7 (CheXmask 권장)
2. **p10 필터**: 우리 학습 데이터 p10 그룹만 추출
3. **Split 매핑**: train/validate/test 분리

> **메모리 주의**: 전체 CSV 로드에 RAM ~8-12GB 필요. ml.t3.xlarge(16GB) 이상 권장.

In [ ]:
%%time

print("=" * 60)
print("전체 CSV 로드 중 (4.4 GB)... 2~5분 소요")
print("=" * 60)

df = pd.read_csv(CHEXMASK_CSV)
print(f"\n전체 행: {len(df):,}")
print(f"컬럼: {list(df.columns)}")
print(f"메모리 사용: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

In [ ]:
# 7-1: 품질 필터 (Dice RCA Mean ≥ 0.7)
dice_col = [c for c in df.columns if 'dice' in c.lower() and 'mean' in c.lower()]
if dice_col:
    dice_col = dice_col[0]
    before = len(df)
    df = df[df[dice_col] >= 0.7].copy()
    print(f"품질 필터 ({dice_col} ≥ 0.7): {before:,} → {len(df):,} ({before - len(df):,}개 제거)")
else:
    print("Dice RCA Mean 컬럼 없음 — 품질 필터 스킵")
    print(f"사용 가능 컬럼: {list(df.columns)}")

In [ ]:
# 7-2: p10 그룹 필터링
id_col = df.columns[0]  # 첫 번째 컬럼 = ImageID
print(f"ImageID 컬럼: {id_col}")
print(f"샘플: {df[id_col].iloc[0]}")

before = len(df)
df_p10 = df[df[id_col].astype(str).str.contains('/p10/', case=False)].copy()

# '/p10/'가 안 맞으면 다른 패턴 시도
if len(df_p10) == 0:
    print("'/p10/' 패턴 매칭 실패, 'p10' 패턴 시도...")
    df_p10 = df[df[id_col].astype(str).str.contains('p10', case=False)].copy()

print(f"p10 필터: {before:,} → {len(df_p10):,}")

# 메모리 절약: 원본 df 삭제
del df
import gc; gc.collect()
print(f"원본 DataFrame 삭제 (메모리 확보)")

In [ ]:
# 7-3: Split 정보 병합 (train/validate/test)
# CheXmask ImageID에서 dicom_id 추출 (파일명에서 .jpg 제거)
df_p10['dicom_id'] = df_p10[id_col].apply(
    lambda x: os.path.splitext(os.path.basename(str(x)))[0]
)

print(f"dicom_id 샘플: {df_p10['dicom_id'].iloc[0]}")

# Split CSV와 병합
split_df = pd.read_csv(SPLIT_CSV)
df_merged = df_p10.merge(split_df[['dicom_id', 'split']], on='dicom_id', how='inner')

print(f"\nSplit 병합 결과: {len(df_p10):,} → {len(df_merged):,}")
print(f"\nSplit 분포:")
print(df_merged['split'].value_counts())

# 메모리 절약
del df_p10
gc.collect()

## Step 8: RLE 디코딩 + NPZ 저장

모든 p10 이미지의 마스크를 디코딩하여 NPZ 파일로 저장.
각 NPZ에는 `combined`, `left_lung`, `right_lung`, `heart`, `image_id` 포함.

- **combined**: 0=배경, 1=좌폐, 2=우폐, 3=심장 (학습 시 이것만 사용)
- **target_size**: 512x512 (U-Net 입력 해상도)

In [ ]:
%%time

TARGET_SIZE = 512

for split_name in ['train', 'validate', 'test']:
    split_df = df_merged[df_merged['split'] == split_name]
    if len(split_df) == 0:
        print(f"\n[{split_name}] 데이터 없음 — 스킵")
        continue
    
    out_dir = os.path.join(OUTPUT_DIR, split_name)
    os.makedirs(out_dir, exist_ok=True)
    
    total = len(split_df)
    saved = 0
    errors = 0
    
    print(f"\n{'='*50}")
    print(f"[{split_name}] {total:,}개 처리 시작 → {out_dir}")
    print(f"{'='*50}")
    
    for idx, (_, row) in enumerate(split_df.iterrows()):
        try:
            combined, left_lung, right_lung, heart = decode_masks(row, TARGET_SIZE)
            
            # 파일명: dicom_id 사용
            dicom_id = row['dicom_id']
            npz_path = os.path.join(out_dir, f"{dicom_id}.npz")
            
            np.savez_compressed(
                npz_path,
                combined=combined,
                left_lung=left_lung,
                right_lung=right_lung,
                heart=heart,
                image_id=str(row[id_col]),
            )
            saved += 1
            
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"  [오류] idx={idx}, dicom={row.get('dicom_id','?')}: {e}")
        
        # 진행률 (500개마다 + 마지막)
        if (idx + 1) % 500 == 0 or idx == total - 1:
            pct = (idx + 1) / total * 100
            print(f"  진행: {idx+1:,}/{total:,} ({pct:.1f}%) | 저장: {saved:,} | 오류: {errors}")
    
    print(f"[{split_name}] 완료: {saved:,}개 저장, {errors}개 오류")

print(f"\n전체 처리 완료!")

## Step 9: 결과 확인 + 샘플 시각화

In [ ]:
# 각 split별 파일 수 확인
print("=== NPZ 파일 수 ===")
total_files = 0
for split_name in ['train', 'validate', 'test']:
    split_dir = os.path.join(OUTPUT_DIR, split_name)
    if os.path.exists(split_dir):
        count = len([f for f in os.listdir(split_dir) if f.endswith('.npz')])
        total_files += count
        # 디스크 사용량
        size_mb = sum(
            os.path.getsize(os.path.join(split_dir, f)) 
            for f in os.listdir(split_dir) if f.endswith('.npz')
        ) / (1024**2)
        print(f"  {split_name}: {count:,}개 ({size_mb:.1f} MB)")
    else:
        print(f"  {split_name}: 디렉토리 없음")
print(f"  총 합계: {total_files:,}개")

In [ ]:
# 랜덤 샘플 5장 시각화
train_dir = os.path.join(OUTPUT_DIR, 'train')
npz_files = [f for f in os.listdir(train_dir) if f.endswith('.npz')]

if len(npz_files) >= 5:
    samples = np.random.choice(npz_files, 5, replace=False)
else:
    samples = npz_files

fig, axes = plt.subplots(len(samples), 4, figsize=(20, 5 * len(samples)))
if len(samples) == 1:
    axes = [axes]

for i, fname in enumerate(samples):
    data = np.load(os.path.join(train_dir, fname))
    
    axes[i][0].imshow(data['combined'], cmap='nipy_spectral', vmin=0, vmax=3)
    axes[i][0].set_title(f'Combined')
    
    axes[i][1].imshow(data['left_lung'], cmap='gray')
    axes[i][1].set_title('Left Lung')
    
    axes[i][2].imshow(data['right_lung'], cmap='gray')
    axes[i][2].set_title('Right Lung')
    
    axes[i][3].imshow(data['heart'], cmap='gray')
    axes[i][3].set_title('Heart')
    
    for ax in axes[i]:
        ax.axis('off')
    
    axes[i][0].set_ylabel(fname[:20], fontsize=10, rotation=0, labelpad=80)

plt.suptitle('Sample Masks (512x512)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'sample_masks_grid.png'), dpi=100, bbox_inches='tight')
plt.show()
print("저장: sample_masks_grid.png")

## Step 10: CTR 테스트 계산 (마스크 검증)

마스크가 정상이면 CTR(Cardiothoracic Ratio) 계산이 가능해야 함.
- **CTR = 심장 가로폭 / 흉곽 가로폭**
- CTR > 0.50 → Cardiomegaly (심비대)

샘플 1장으로 CTR을 계산하여 마스크 품질을 간접 검증.

In [ ]:
def calculate_ctr(combined_mask):
    """
    Combined 마스크에서 CTR 계산
    
    CTR = 심장 가로폭 / 흉곽 가로폭
    
    흉곽 가로폭 = 좌폐 최좌측 ~ 우폐 최우측
    심장 가로폭 = 심장 마스크의 가로 범위
    """
    heart_mask = (combined_mask == 3)
    lung_mask = (combined_mask == 1) | (combined_mask == 2)
    
    if not heart_mask.any() or not lung_mask.any():
        return None, None, None
    
    # 심장 가로폭: 심장이 있는 열(column)의 범위
    heart_cols = np.where(heart_mask.any(axis=0))[0]
    heart_width = heart_cols[-1] - heart_cols[0]
    
    # 흉곽 가로폭: 폐가 있는 열의 범위
    lung_cols = np.where(lung_mask.any(axis=0))[0]
    thorax_width = lung_cols[-1] - lung_cols[0]
    
    ctr = heart_width / thorax_width if thorax_width > 0 else 0
    
    return ctr, heart_width, thorax_width


# 샘플 3장으로 CTR 테스트
print("=== CTR 테스트 계산 ===")
print(f"{'파일':<30} {'심장폭':>8} {'흉곽폭':>8} {'CTR':>8} {'판정':>12}")
print("-" * 70)

test_files = npz_files[:3] if len(npz_files) >= 3 else npz_files
for fname in test_files:
    data = np.load(os.path.join(train_dir, fname))
    ctr, hw, tw = calculate_ctr(data['combined'])
    
    if ctr is not None:
        diagnosis = "Cardiomegaly" if ctr > 0.50 else "Normal"
        print(f"{fname[:28]:<30} {hw:>8} {tw:>8} {ctr:>8.3f} {diagnosis:>12}")
    else:
        print(f"{fname[:28]:<30} {'N/A':>8} {'N/A':>8} {'N/A':>8} {'마스크 없음':>12}")

print("\nCTR > 0.50 → Cardiomegaly (심비대)")
print("CTR 정상 범위: 0.42 ~ 0.50")

## Step 11: S3 업로드

NPZ 마스크 파일을 S3에 업로드. U-Net 학습 시 S3에서 직접 로드.

In [ ]:
%%time

print(f"S3 업로드: {OUTPUT_DIR} → s3://{BUCKET}/{S3_DATA_PREFIX}/")
!aws s3 sync {OUTPUT_DIR} s3://{BUCKET}/{S3_DATA_PREFIX}/ --quiet

print("\n업로드 완료!")
# 확인
!aws s3 ls s3://{BUCKET}/{S3_DATA_PREFIX}/ --summarize --recursive | tail -3

## Step 12: 정리 (선택사항)

디스크 공간 확보가 필요하면 CheXmask CSV(4.4GB)를 삭제.
NPZ 파일은 S3에 올렸으므로 로컬도 삭제 가능.

In [ ]:
# 디스크 사용량 확인
!du -sh {WORK_DIR}/*

# 필요시 CheXmask CSV 삭제 (4.4GB 확보)
# import os
# os.remove(CHEXMASK_CSV)
# print("CheXmask CSV 삭제 완료")

## 완료!

**생성된 데이터**:
```
s3://{BUCKET}/data/unet_masks/
  ├── train/       (~8,000+ NPZ)
  ├── validate/    (~60+ NPZ)
  └── test/        (~60+ NPZ)
```

**각 NPZ 파일 내용**:
- `combined`: (512, 512) uint8 — 0=배경, 1=좌폐, 2=우폐, 3=심장
- `left_lung`, `right_lung`, `heart`: (512, 512) uint8 — 개별 바이너리 마스크
- `image_id`: str — 원본 MIMIC-CXR 이미지 참조

**다음 단계**: `02_train_unet.ipynb`에서 U-Net 학습